# 09.5 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/05-hugging-face/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

## Exercise 1: memory by hand

How much memory do these need in float32 and float16? DistilBERT (66 million parameters), BERT-base (110 million), a 3-billion-parameter LLM, a 7-billion-parameter LLM. Which fit in 8 GB of RAM?

In [ ]:
# your code here

## Exercise 2: write your own topics

Add a topic *"praise"* and two emails that thank ShopWave. Then change the hypothesis template (the `hypothesis_template` argument, default `"This example is {}."`) to `"This customer email is about {}."`. Does accuracy change?

In [ ]:
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

import torch, time
from transformers import pipeline

device = "mps" if torch.backends.mps.is_available() else (0 if torch.cuda.is_available() else -1)
print("running on", {-1: "CPU", 0: "NVIDIA GPU", "mps": "Apple GPU"}[device])

emails = [
    "Hi, my order 18734 still hasn't arrived and it's been two weeks. Where is it?",
    "I'd like my money back for the lantern, it doesn't fit my shelf.",
    "Is the ceramic teapot dishwasher safe?",
    "The cake stand came smashed and nobody answered my last two emails. Very poor service.",
    "Can you tell me if the fairy lights work outdoors?",
    "Please refund the £24.50 for the cancelled doormat.",
    "Tracking says delivered but there is nothing at my door.",
    "Your courier left the parcel in the rain and everything inside is ruined. Unacceptable.",
]
truth = ["delivery", "refund", "product question", "complaint",
         "product question", "refund", "delivery", "complaint"]
topics = ["delivery", "refund", "product question", "complaint"]

def route(model_name):
    clf = pipeline("zero-shot-classification", model=model_name, device=device)
    start = time.time()
    out = clf(emails, candidate_labels=topics)
    pred = [o["labels"][0] for o in out]
    correct = sum(p == t for p, t in zip(pred, truth))
    return clf, out, pred, correct, time.time() - start

In [ ]:
# your code here

## Exercise 3: is bigger better?

Route the 8 emails with `MoritzLaurer/deberta-v3-base-zeroshot-v2.0` (about 740 MB download). Compare accuracy, time and parameter count with the lesson's model.

In [ ]:
# your code here

## Exercise 4: tokens for ShopWave's own words

Tokenise ten ShopWave product names. Which words get split into the most pieces? Why might that matter?

In [ ]:
# your code here